# Train/Test Splitting

Loads the unified dataset from `data/combined_dataset.mat` and splits each file's signal by time (not by file) into train/test — so a later windowing step never puts samples from the same continuous recording in both train and test, which would leak information.

In [1]:
import os

import numpy as np
import pandas as pd
import scipy.io as sio

DATA_DIR = "data"
SAMPLE_RATE_HZ = 48_000  # all files here are from CWRU's 48k drive-end fault dataset
combined_path = os.path.join(DATA_DIR, "combined_dataset.mat")

# simplify_cells collapses each MATLAB struct into a plain dict and squeezes
# the signal arrays down to 1D, so no manual [0, 0]/.dtype.names unpacking is needed.
combined = sio.loadmat(combined_path, simplify_cells=True)

df = pd.DataFrame(
    {"var_name": var_name, **entry}
    for var_name, entry in combined.items()
    if not var_name.startswith("__")
)

print(f"{len(df)} files loaded from {combined_path}")
df.head()

56 files loaded from data/combined_dataset.mat


,var_name,filename,DE_time,FE_time,BA_time,rpm_reported,category,fault_location,fault_diameter_in,load_hp,rpm,position,file_number
0,fault_122,48k_drive_end_fault_ball_0.007in_0hp_1797rpm_1...,"[-0.111192, -0.08302892307692307, -0.042348923...","[-0.09512545454545453, -0.07211454545454546, -...",[],1796.0,fault,ball,0.007,0,1797.0,[],122.0
1,fault_123,48k_drive_end_fault_ball_0.007in_1hp_1772rpm_1...,"[-0.04109723076923077, -0.046104, -0.028371692...","[-0.08115454545454545, -0.09759090909090909, -...",[],1772.0,fault,ball,0.007,1,1772.0,[],123.0
2,fault_124,48k_drive_end_fault_ball_0.007in_2hp_1750rpm_1...,"[0.09992676923076924, 0.14164984615384615, 0.1...","[-0.0741690909090909, -0.04972, -0.02198363636...",[],1747.0,fault,ball,0.007,2,1750.0,[],124.0
3,fault_125,48k_drive_end_fault_ball_0.007in_3hp_1730rpm_1...,"[0.25305046153846156, 0.2574313846153846, 0.23...","[0.0538290909090909, 0.12265636363636362, 0.16...",[],1721.0,fault,ball,0.007,3,1730.0,[],125.0
4,fault_189,48k_drive_end_fault_ball_0.014in_0hp_1797rpm_1...,"[0.03984553846153846, 0.0897046153846154, 0.13...","[0.039241818181818176, 0.08834545454545455, 0....",[],1797.0,fault,ball,0.014,0,1797.0,[],189.0


## Split each file's signal by time

Splitting is done per file, before any windowing: the first `train_frac` of each recording goes to train, the rest to test. This keeps every train/test window confined to one side of the split, so no window straddles the boundary or pulls from what should be held-out data.

In [2]:
def split_signal_train_test(signal, train_frac=0.8):
    split_point = int(len(signal) * train_frac)
    train_signal = signal[:split_point]
    test_signal = signal[split_point:]
    return train_signal, test_signal

In [3]:
SIGNAL_COLUMNS = ["DE_time", "FE_time", "BA_time"]

train_rows = []
test_rows = []

for _, row in df.iterrows():
    train_row = row.drop(labels=SIGNAL_COLUMNS).to_dict()
    test_row = train_row.copy()
    for col in SIGNAL_COLUMNS:
        signal = row[col]
        train_signal, test_signal = split_signal_train_test(signal)
        train_row[col] = train_signal
        test_row[col] = test_signal
    train_rows.append(train_row)
    test_rows.append(test_row)

train_df = pd.DataFrame(train_rows)
test_df = pd.DataFrame(test_rows)

print(f"train_df: {len(train_df)} files, test_df: {len(test_df)} files")

train_df: 56 files, test_df: 56 files


In [4]:
# Sanity check: train + test should reconstruct the original signal for every file, with no overlap
for i in range(len(df)):
    original = df.iloc[i]["DE_time"]
    train_signal = train_df.iloc[i]["DE_time"]
    test_signal = test_df.iloc[i]["DE_time"]
    assert len(train_signal) + len(test_signal) == len(original)
    assert np.array_equal(np.concatenate([train_signal, test_signal]), original)

sample = df.iloc[0]
print(f"e.g. {sample['filename']}: total={len(sample['DE_time'])}, "
      f"train={len(train_df.iloc[0]['DE_time'])}, test={len(test_df.iloc[0]['DE_time'])}")

e.g. 48k_drive_end_fault_ball_0.007in_0hp_1797rpm_122.mat: total=244739, train=195791, test=48948


## Window the split signals, organized by load

Slice each file's already time-split `DE_time` (train and test separately, from `train_df`/`test_df` above) into fixed-size, non-overlapping windows, and group them by `load_hp` rather than committing to one fixed source/target pair.

`make_splits(source_load, target_load)` then builds the 4-bucket domain-adaptation dict for any single (source, target) pair on demand — a file's train-split windows become `source_train`/`target_labeled`, its test-split windows become `source_test`/`target_test` — so all 12 ordered pairs among `{0, 1, 2, 3}` hp can be evaluated (e.g. 0→1, 0→2, ..., 3→2, averaging results) without re-windowing the signals for each pair.

In [5]:
def segment_signal(signal, window_size=4096):
    """Slice a 1D signal into non-overlapping fixed-size windows; leftover samples that don't fill a full window are dropped."""
    n_windows = len(signal) // window_size
    if n_windows == 0:
        return np.empty((0, window_size))
    return signal[: n_windows * window_size].reshape(n_windows, window_size)


def make_splits(source_load: int, target_load: int) -> dict:
    """Build the 4-bucket domain-adaptation split for one (source_load -> target_load) pair."""
    return {
        "source_train": windows_by_load[source_load]["train"],
        "source_test": windows_by_load[source_load]["test"],
        "target_labeled": windows_by_load[target_load]["train"],
        "target_test": windows_by_load[target_load]["test"],
    }

In [6]:
import pickle

WINDOW_SIZE = 4096
SIGNAL_COL = "DE_time"
META_COLUMNS = [c for c in df.columns if c not in SIGNAL_COLUMNS]

# windows_by_load[load_hp]["train"/"test"] = list of {**metadata, "window": w}
windows_by_load = {load: {"train": [], "test": []} for load in sorted(df["load_hp"].unique())}

for i in range(len(df)):
    train_row = train_df.iloc[i]
    test_row = test_df.iloc[i]
    meta = train_row[META_COLUMNS].to_dict()  # metadata is identical between the train/test rows of a file
    load = meta["load_hp"]

    train_windows = segment_signal(train_row[SIGNAL_COL], WINDOW_SIZE)
    test_windows = segment_signal(test_row[SIGNAL_COL], WINDOW_SIZE)

    windows_by_load[load]["train"].extend({**meta, "window": w} for w in train_windows)
    windows_by_load[load]["test"].extend({**meta, "window": w} for w in test_windows)

for load, load_windows in windows_by_load.items():
    print(f"load_hp={load}: train={len(load_windows['train'])}, test={len(load_windows['test'])} windows")

# One consolidated file, not 12 — make_splits() builds each pair's dict on demand from this,
# instead of duplicating every load's windows across the pairs it appears in.
windows_path = os.path.join(DATA_DIR, "windows_by_load.pkl")
with open(windows_path, "wb") as f:
    pickle.dump(windows_by_load, f)
print(f"Saved to {windows_path}")

load_hp=0: train=536, test=128 windows
load_hp=1: train=1322, test=322 windows
load_hp=2: train=1326, test=322 windows
load_hp=3: train=1323, test=322 windows


Saved to data/windows_by_load.pkl


In [7]:
# Sanity check: build the split for all 12 ordered (source, target) pairs and confirm window shapes
from itertools import permutations

LOADS = sorted(windows_by_load.keys())

for source_load, target_load in permutations(LOADS, 2):
    pair_splits = make_splits(source_load, target_load)
    counts = {key: len(items) for key, items in pair_splits.items()}
    assert all(item["window"].shape == (WINDOW_SIZE,) for items in pair_splits.values() for item in items)
    print(f"{source_load}->{target_load}: {counts}")

0->1: {'source_train': 536, 'source_test': 128, 'target_labeled': 1322, 'target_test': 322}
0->2: {'source_train': 536, 'source_test': 128, 'target_labeled': 1326, 'target_test': 322}
0->3: {'source_train': 536, 'source_test': 128, 'target_labeled': 1323, 'target_test': 322}
1->0: {'source_train': 1322, 'source_test': 322, 'target_labeled': 536, 'target_test': 128}
1->2: {'source_train': 1322, 'source_test': 322, 'target_labeled': 1326, 'target_test': 322}
1->3: {'source_train': 1322, 'source_test': 322, 'target_labeled': 1323, 'target_test': 322}
2->0: {'source_train': 1326, 'source_test': 322, 'target_labeled': 536, 'target_test': 128}
2->1: {'source_train': 1326, 'source_test': 322, 'target_labeled': 1322, 'target_test': 322}
2->3: {'source_train': 1326, 'source_test': 322, 'target_labeled': 1323, 'target_test': 322}
3->0: {'source_train': 1323, 'source_test': 322, 'target_labeled': 536, 'target_test': 128}
3->1: {'source_train': 1323, 'source_test': 322, 'target_labeled': 1322, 'ta

## Feature extraction

Four extraction methods, each applied to every window in `windows_by_load`:

1. **Raw time domain** — the window itself; captures amplitude patterns directly.
2. **FFT magnitude spectrum** — captures frequency content, but bearing fault impacts are usually buried under broadband structural resonance here.
3. **Envelope spectrum** — `|Hilbert(window)|`, then FFT of that envelope. Demodulates the signal so the fault impact *rate* shows up as clean spectral lines, isolated from the high-frequency carrier resonance.
4. **Fault characteristic peaks (BPFO/BPFI/BSF)** — the envelope spectrum's magnitude at the theoretical outer-race/inner-race/ball-spin fault frequencies (and their 2nd/3rd harmonics), computed from each window's RPM. This compresses the dense envelope spectrum into a small, physically-grounded feature vector.

Saved as **`.npz`**, not `.pkl`: these are numeric feature matrices meant to feed a model (unlike `windows_by_load.pkl`, which mixes raw signals with per-window metadata dicts) — `npz` loads faster, is more compact, and doesn't unpickle arbitrary Python objects. Metadata is kept alongside as parallel named arrays in the same file, and all four files share the same row order, so row `i` refers to the same window across every feature file.

In [8]:
# Characteristic fault frequencies as multiples of shaft speed, for the CWRU drive-end
# bearing (SKF 6205-2RS JEM) — the standard values cited across CWRU-based fault diagnosis work.
FAULT_FREQ_ORDERS = {
    "BPFO": 3.5848,  # ball pass frequency, outer race
    "BPFI": 5.4152,  # ball pass frequency, inner race
    "BSF": 2.357,    # ball spin frequency
}

# Nominal shaft speed by load, used only as a fallback when a file has no recorded RPM
# (normal_1hp.mat / normal_2hp.mat are missing the RPM channel — see data_download.ipynb).
NOMINAL_RPM_BY_LOAD = {0: 1797, 1: 1772, 2: 1750, 3: 1730}


def resolve_rpm(item):
    rpm = item.get("rpm_reported")
    if rpm is None or (isinstance(rpm, float) and np.isnan(rpm)):
        return NOMINAL_RPM_BY_LOAD[item["load_hp"]]
    return rpm


def fault_frequencies_hz(rpm):
    """Theoretical BPFO/BPFI/BSF frequencies (Hz) for a given shaft speed (RPM)."""
    shaft_hz = rpm / 60.0
    return {name: order * shaft_hz for name, order in FAULT_FREQ_ORDERS.items()}

In [9]:
from scipy.signal import hilbert

FFT_FREQS = np.fft.rfftfreq(WINDOW_SIZE, d=1 / SAMPLE_RATE_HZ)
N_HARMONICS = 3
FAULT_FREQ_TOL_HZ = 5.0  # tolerance window when reading off a peak near a theoretical fault frequency
FAULT_FREQ_FEATURE_NAMES = [
    f"{name}_h{h}" for name in FAULT_FREQ_ORDERS for h in range(1, N_HARMONICS + 1)
]


def fft_magnitude(window):
    """FFT magnitude spectrum, normalized by window length."""
    return np.abs(np.fft.rfft(window)) / len(window)


def envelope_spectrum(window):
    """Magnitude spectrum of the signal's Hilbert envelope (amplitude demodulation)."""
    envelope = np.abs(hilbert(window))
    envelope = envelope - envelope.mean()  # drop DC so it doesn't dominate the spectrum
    return np.abs(np.fft.rfft(envelope)) / len(envelope)


def fault_freq_peaks(envelope_mag, rpm):
    """Peak envelope-spectrum magnitude near each fault frequency's 1st-3rd harmonics."""
    freqs_hz = fault_frequencies_hz(rpm)
    peaks = []
    for base_freq in freqs_hz.values():
        for h in range(1, N_HARMONICS + 1):
            mask = np.abs(FFT_FREQS - base_freq * h) <= FAULT_FREQ_TOL_HZ
            peaks.append(envelope_mag[mask].max() if mask.any() else 0.0)
    return np.array(peaks)

In [10]:
META_FIELDS = ["var_name", "filename", "category", "fault_location", "fault_diameter_in",
               "load_hp", "file_number", "rpm_reported"]

time_rows, fft_rows, envelope_rows, fault_freq_rows = [], [], [], []
meta_records = []

for load, load_windows in windows_by_load.items():
    for split in ("train", "test"):
        for item in load_windows[split]:
            window = item["window"]
            rpm = resolve_rpm(item)

            fft_mag = fft_magnitude(window)
            env_mag = envelope_spectrum(window)

            time_rows.append(window)
            fft_rows.append(fft_mag)
            envelope_rows.append(env_mag)
            fault_freq_rows.append(fault_freq_peaks(env_mag, rpm))

            meta_records.append({
                **{field: item.get(field) for field in META_FIELDS},
                "split": split,
                "rpm_resolved": rpm,
            })

meta_df = pd.DataFrame(meta_records)

features_time = np.stack(time_rows)
features_fft = np.stack(fft_rows)
features_envelope = np.stack(envelope_rows)
features_fault_freq = np.stack(fault_freq_rows)

print(f"{len(meta_df)} windows total")
print(f"features_time:       {features_time.shape}")
print(f"features_fft:        {features_fft.shape}")
print(f"features_envelope:   {features_envelope.shape}")
print(f"features_fault_freq: {features_fault_freq.shape}  ({FAULT_FREQ_FEATURE_NAMES})")

5601 windows total
features_time:       (5601, 4096)
features_fft:        (5601, 2049)
features_envelope:   (5601, 2049)
features_fault_freq: (5601, 9)  (['BPFO_h1', 'BPFO_h2', 'BPFO_h3', 'BPFI_h1', 'BPFI_h2', 'BPFI_h3', 'BSF_h1', 'BSF_h2', 'BSF_h3'])


In [11]:
def save_features(path, X, meta_df, **extra_arrays):
    """Save a feature matrix plus per-row metadata (as parallel named arrays) into one .npz."""
    arrays = {"X": X, **extra_arrays}
    for col in meta_df.columns:
        arrays[col] = meta_df[col].to_numpy()
    np.savez_compressed(path, **arrays)


save_features(os.path.join(DATA_DIR, "features_time.npz"), features_time, meta_df)
save_features(os.path.join(DATA_DIR, "features_fft.npz"), features_fft, meta_df, freqs_hz=FFT_FREQS)
save_features(os.path.join(DATA_DIR, "features_envelope.npz"), features_envelope, meta_df, freqs_hz=FFT_FREQS)
save_features(
    os.path.join(DATA_DIR, "features_fault_freq.npz"), features_fault_freq, meta_df,
    feature_names=np.array(FAULT_FREQ_FEATURE_NAMES),
)

for name in ["features_time", "features_fft", "features_envelope", "features_fault_freq"]:
    path = os.path.join(DATA_DIR, f"{name}.npz")
    print(f"Saved {path} ({os.path.getsize(path) / 1e6:.1f} MB)")

Saved data/features_time.npz (65.5 MB)
Saved data/features_fft.npz (87.4 MB)
Saved data/features_envelope.npz (87.6 MB)
Saved data/features_fault_freq.npz (0.3 MB)


In [12]:
# Sanity check: reload from disk, confirm shapes, and confirm the 4 files stay row-aligned
reloaded = {
    name: np.load(os.path.join(DATA_DIR, f"{name}.npz"), allow_pickle=True)
    for name in ["features_time", "features_fft", "features_envelope", "features_fault_freq"]
}

for name, npz in reloaded.items():
    print(f"{name}: X={npz['X'].shape}, load_hp[:5]={npz['load_hp'][:5]}, split[:5]={npz['split'][:5]}")

assert all(np.array_equal(reloaded["features_time"]["filename"], npz["filename"]) for npz in reloaded.values())
print("\nRow order matches across all 4 feature files.")

features_time: X=(5601, 4096), load_hp[:5]=[0 0 0 0 0], split[:5]=['train' 'train' 'train' 'train' 'train']


features_fft: X=(5601, 2049), load_hp[:5]=[0 0 0 0 0], split[:5]=['train' 'train' 'train' 'train' 'train']


features_envelope: X=(5601, 2049), load_hp[:5]=[0 0 0 0 0], split[:5]=['train' 'train' 'train' 'train' 'train']
features_fault_freq: X=(5601, 9), load_hp[:5]=[0 0 0 0 0], split[:5]=['train' 'train' 'train' 'train' 'train']

Row order matches across all 4 feature files.
